# 02 预测解析与匿名化

统一解析 DeepGO JSON 与 NetGO TXT 两种预测产物，并说明匿名 ID 映射。

## Entry 提取规则
- DeepGO JSON：`protein_info` 可能是完整 FASTA 头（如 `sp|Q7Z020|TRPA1_DROME ...`）或纯 accession。
  统一取 `sp|` / `tr|` 之后、下一个 `|` 之前的文字作为 **Entry**。
- NetGO TXT：第 1 列可能是 `sp|Q7Z020|TRPA1` 或纯 accession，同样按上述规则提取。


In [ ]:
# -*- coding: utf-8 -*-
import re, json, csv, os

ACC_RE = re.compile(r"(?:sp|tr)\\|([^|]+)\\|")   # sp|Entry|... 或 tr|Entry|...

def acc_of(pi):
    """从 protein_info / entry 文本提取 Entry（sp| 后、下一 | 前）"""
    m = ACC_RE.match(pi or "")
    return m.group(1) if m else pi

def load_deepgo_json(fp):
    """DeepGO JSON -> {Entry: {GO_id: score}}（保留全部 GO）"""
    out = {}
    with open(fp, encoding="utf-8") as fh:
        data = json.load(fh)
    for batch in data:
        for pred in batch.get("data", {}).get("predictions", []):
            acc = acc_of(pred.get("protein_info", ""))
            gs = {}
            for cat_block in pred.get("functions", []):
                for fn in cat_block.get("functions", []):
                    if len(fn) >= 3:
                        gs[fn[0]] = float(fn[2])
            out[acc] = gs
    return out

def load_netgo_txt(fp):
    """NetGO txt -> {Entry: {GO_id: score}}；txt 列为 entry\tGO\tscore\tcat\tterm"""
    out = {}
    with open(fp, encoding="utf-8") as fh:
        for ln in fh:
            parts = ln.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            key = acc_of(parts[0])
            try:
                sc = float(parts[2])
            except ValueError:
                continue
            out.setdefault(key, {})[parts[1]] = sc
    return out

# 匿名映射（本仓库提供 entry_anon_mapping.csv）：
# 列 File / Entry / Anon_ID / Original_Header，Anon_ID 为全局唯一匿名编号。
def load_anon_mapping(path="../data/entry_anon_mapping.csv"):
    m = {}
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            m.setdefault(r["File"], {})[r["Entry"]] = r["Anon_ID"]
    return m


### 说明
- `entry_anon_mapping.csv` 覆盖全部输入 FASTA（2026 easy、5 个随机种子 test/train、trEMBL 5551），共 11080 行。
- 匿名 FASTA 文件（`*_anon.fasta`）与匿名预测结果（`result_test*_anon.txt`、`result_train42_anon.txt`、`result_redundant_predictions.txt`）用于在不泄露 accession 的前提下复现指标。
